In [1]:
# ============================================================
# LEVEL 4:
# GEMINI + LANGCHAIN AGENT + MULTIPLE TOOLS
# + RAG + EMBEDDINGS + CHROMADB + MEMORY
#
# GOOGLE COLAB - SINGLE CELL
# ============================================================

# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q \
    langchain \
    langchain-google-genai \
    langchain-chroma \
    langgraph \
    sentence-transformers \
    chromadb \
    wikipedia

In [2]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import math
import wikipedia
from google.colab import userdata
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

In [3]:
# ============================================================
# 3. GEMINI API KEY
# ============================================================

# In Google Colab:
#
# Left panel
#      ↓
# Secrets
#      ↓
# Add:
#
# GEMINI_API_KEY

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [4]:
# ============================================================
# 4. CREATE GEMINI LLM
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)
print("Gemini LLM initialized.")

# ============================================================
# 5. CREATE CUSTOM EMBEDDING CLASS
# ============================================================

class MiniLMEmbeddings(Embeddings):

    def __init__(self):
        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

    def embed_documents(self, texts):
        embeddings = self.model.encode(
            texts
        )
        return embeddings.tolist()

    def embed_query(self, text):
        embedding = self.model.encode(
            text
        )
        return embedding.tolist()

# Create embedding model

embedding_model = MiniLMEmbeddings()
print(
    "Embedding model loaded:"
    " all-MiniLM-L6-v2"
)

# ============================================================
# 6. SAMPLE KNOWLEDGE BASE
# ============================================================

documents = [
    """
    Retrieval-Augmented Generation, commonly called RAG,
    combines information retrieval with a large language
    model. A RAG system retrieves relevant information
    from an external knowledge source and supplies that
    information to the LLM as context before generating
    the final response.
    """,

    """
    ChromaDB is an open-source vector database.
    It stores embeddings and supports vector similarity
    search. ChromaDB is commonly used in RAG systems
    for finding documents that are semantically similar
    to a user's query.
    """,

    """
    Embeddings convert text into numerical vectors.
    Semantically similar sentences usually produce
    vectors that are close together in vector space.
    Common similarity methods include cosine similarity,
    dot product and Euclidean distance.
    """,

    """
    Gemini is Google's family of multimodal large
    language models. Gemini models can process text,
    images, audio and other types of information,
    depending on the selected model.
    """,

    """
    AI agents combine a language model with tools.
    An agent can reason about a user's request,
    decide which tool should be used, execute that
    tool and then use the result to continue reasoning.
    """,

    """
    Agent memory allows an AI system to retain
    information from earlier interactions. Short-term
    memory usually stores conversation history within
    a session, while long-term memory can persist
    information across different sessions.
    """,

    """
    Physical AI combines artificial intelligence
    with systems that sense and interact with the
    physical world. Examples include robots,
    autonomous vehicles, drones and smart
    manufacturing systems.
    """,

    """
    A vector database is optimized for storing and
    retrieving high-dimensional numerical vectors.
    Vector databases are widely used for semantic
    search, recommendation systems and RAG.
    """

]

Gemini LLM initialized.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: all-MiniLM-L6-v2


In [5]:
# ============================================================
# 7. CREATE CHROMADB VECTOR STORE
# ============================================================

vector_store = Chroma.from_texts(
    texts=documents,
    embedding=embedding_model,
    collection_name="level4_knowledge_base"
)
print(
    "Knowledge documents stored in ChromaDB."
)

Knowledge documents stored in ChromaDB.


In [6]:
# ============================================================
# 8. CREATE RAG TOOL
# ============================================================

@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the private ChromaDB knowledge base using
    vector similarity search. Use this tool for questions
    about RAG, embeddings, ChromaDB, AI agents, memory,
    Gemini or Physical AI.
    """
    results = vector_store.similarity_search(
        query,
        k=3
    )

    if not results:
        return (
            "No relevant information was found "
            "in the knowledge base."
        )


    context = "\n\n".join(
        [
            document.page_content.strip()
            for document in results
        ]
    )
    return f"""
Relevant information from the knowledge base:
{context}
"""


# ============================================================
# 9. WIKIPEDIA TOOL
# ============================================================

@tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for general factual knowledge.
    Use this tool when the answer is not available
    in the private RAG knowledge base.
    """
    try:
        results = wikipedia.search(
            query,
            results=3
        )

        if not results:
            return (
                "No Wikipedia results found."
            )

        title = results[0]


        try:

            page = wikipedia.page(
                title,
                auto_suggest=False
            )


        except wikipedia.DisambiguationError as e:
            title = e.options[0]
            page = wikipedia.page(
                title,
                auto_suggest=False
            )


        # Limit size for demo
        content = page.content[:6000]
        return f"""
Wikipedia article:
{page.title}
Content:
{content}
"""

    except Exception as e:
        return (
            "Wikipedia search failed: "
            + str(e)
        )

# ============================================================
# 10. CALCULATOR TOOL
# ============================================================

@tool
def calculator(expression: str) -> str:
    """
    Perform simple mathematical calculations.
    Input should be a mathematical expression,
    for example:
    25 * 18
    sqrt(144)
    5000 * 0.08
    """

    try:
        allowed_names = {
            "sqrt":
                math.sqrt,
            "sin":
                math.sin,
            "cos":
                math.cos,
            "tan":
                math.tan,
            "log":
                math.log,
            "pi":
                math.pi,
            "e":
                math.e
        }


        result = eval(
            expression,
            {
                "__builtins__":
                    {}
            },
            allowed_names
        )
        return str(result)

    except Exception as e:
        return (
            "Calculation error: "
            + str(e)
        )

# ============================================================
# 11. SIMPLE CURRENT-TIME TOOL
# ============================================================

@tool
def get_system_information(query: str) -> str:
    """
    Return simple information about this demonstration system.
    Use when asked what components or technologies the system uses.
    """

    return """
This Level-4 AI system contains:

1. Gemini LLM
2. LangChain Agent
3. SentenceTransformer embeddings
4. ChromaDB vector database
5. RAG knowledge retrieval
6. Wikipedia external knowledge tool
7. Calculator tool
8. Conversation memory
"""


# ============================================================
# 12. CREATE MULTIPLE TOOLS
# ============================================================

tools = [
    search_knowledge_base,
    search_wikipedia,
    calculator,
    get_system_information
]

print(
    "Tools created:"
)
for t in tools:
    print(
        "-",
        t.name
    )

Tools created:
- search_knowledge_base
- search_wikipedia
- calculator
- get_system_information


In [7]:
# ============================================================
# 13. CREATE MEMORY
# ============================================================
memory = InMemorySaver()
print(
    "Conversation memory initialized."
)

# ============================================================
# 14. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """

You are an intelligent Level-4 AI assistant.
You have access to multiple tools.
Your job is to decide which tool or tools should
be used based on the user's request.

TOOLS:

1. search_knowledge_base
Use this FIRST when the user asks about:

- RAG
- embeddings
- vector databases
- ChromaDB
- Gemini
- agents
- agent memory
- Physical AI

This is the private RAG knowledge base.

2. search_wikipedia

Use this for general factual questions when
information is not available in the private
knowledge base.

3. calculator

Use this for mathematical calculations.

4. get_system_information

Use this when the user asks about the architecture
or components of this AI application.

IMPORTANT RULES:
- Do not invent information.
- Prefer the RAG knowledge base for topics available there.
- Use tools when useful.
- You may use more than one tool for a question.
- Combine tool results into a clear final answer.
- Remember information from earlier messages in the
  same conversation.
- If the user refers to something discussed earlier,
  use your conversation memory.

"""

# ============================================================
# 15. CREATE LANGCHAIN AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory

)
print(
    "\nLangChain agent created successfully."
)

# ============================================================
# 16. SET CONVERSATION THREAD
# ============================================================

# Same thread ID means the agent remembers
# previous messages.

config = {

    "configurable": {
        "thread_id":
            "level4_training_demo"
    }
}

Conversation memory initialized.

LangChain agent created successfully.


In [8]:
# ============================================================
# 17. START INTERACTIVE CHAT
# ============================================================

print()
print(
    "=" * 70
)
print(
    "LEVEL 4 AGENTIC RAG SYSTEM"
)
print(
    "=" * 70
)
print()
print(
    "Features:"
)
print(
    "Gemini + Agent + RAG + ChromaDB + "
    "Embeddings + Tools + Memory"
)
print()
print(
    "Type 'exit' to stop."
)
print()


while True:
    question = input(
        "You: "
    )
    if question.lower() in [
        "exit",
        "quit"

    ]:
        print(
            "Agent stopped."
        )
        break

    # ========================================================
    # RUN AGENT
    # ========================================================

    result = agent.invoke(
        {
            "messages": [
                {
                    "role":
                        "user",
                    "content":
                        question
                }
            ]
        },
        config=config
    )


    # ========================================================
    # FINAL AGENT RESPONSE
    # ========================================================

    final_message = (
        result["messages"][-1]
    )
    print()
    print(
        "Agent:"
    )
    print(
        final_message.content
    )
    print()
    print(
        "-" * 70
    )


LEVEL 4 AGENTIC RAG SYSTEM

Features:
Gemini + Agent + RAG + ChromaDB + Embeddings + Tools + Memory

Type 'exit' to stop.

You: What is Rag?

Agent:
[{'type': 'text', 'text': 'Retrieval-Augmented Generation, or RAG, is a technique that merges information retrieval with a large language model (LLM). In a RAG system, relevant information is fetched from an external knowledge source and provided to the LLM as context before it generates a response. This helps the LLM produce more accurate and informed answers.', 'extras': {'signature': 'CqoFARFNMg+PvQvgweb+TSQ04l/C245x4BEWxPpOd+CEMIGNsT9a63JPZFHYCLMGDAdMM9o9yMMFeCGienGSq5kCVpOVU+dFRzfM80txEEjsZRjJ4WSjlQvdaHH+SvrYPgP+OqgC1uYzHdBwBrEnb4Evb6fVqL9vFh3V86ouO1QUdz2yAvQ4OGd2vQO1p1ydyqXSnVmXPv+DgNW5rZn5Ua/NUfV1a2jfaYHs3B1EbTqYbBYp2r5+eHxJOVTmEhXgfex34X2SAM5QF1feYelZEta0gfF93ZeyhZUrhUpybx10vK3797pqLoyxEzTplRmReRcjqAxcZ2FvKMP0eox24e8+q+uTRMfy49JtCI/PWWQ32MzrAOUGaKmnsScysc9Tla39jEFgDFdCZ4saJ3sHSj0kbxQeXtE6WafDb/ChKkxrcSumW50j2VfNH8yoNz2zvPM1cycVbwS